In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [45]:
TRAIN_FILE = Path("synthetic_greenwash_train.xlsx")   # change if needed

df = pd.read_excel(TRAIN_FILE)
df = df.dropna(subset=["claim_sentence", "label"]).copy()

print("Total training rows:", len(df))
df["label"].value_counts()


Total training rows: 625


label
Low       307
Medium    170
High      148
Name: count, dtype: int64

In [46]:
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

df["label_id"] = df["label"].map(LABEL_MAP)

if df["label_id"].isna().any():
    bad = df[df["label_id"].isna()]["label"].unique()
    raise ValueError(f"Unknown label(s) found: {bad}")

X_text = df["claim_sentence"].astype(str).tolist()
y = df["label_id"].astype(int).to_numpy()

print("Labels encoded ✅")


Labels encoded ✅


In [47]:
from sentence_transformers import SentenceTransformer

SBERT_NAME = "all-MiniLM-L6-v2"

embedder = SentenceTransformer(SBERT_NAME)

X = embedder.encode(X_text, convert_to_numpy=True, show_progress_bar=True)

print("Embeddings shape:", X.shape)


Batches: 100%|██████████| 20/20 [00:03<00:00,  5.76it/s]

Embeddings shape: (625, 384)


In [48]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=5000, class_weight="balanced")
clf.fit(X_train, y_train)

print("Model trained ✅")


Model trained ✅


In [49]:
from sklearn.metrics import classification_report, accuracy_score

pred = clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, pred, target_names=["Low","Medium","High"]))


Accuracy: 0.832

Classification Report:

              precision    recall  f1-score   support

         Low       0.90      0.93      0.92        61
      Medium       0.74      0.74      0.74        34
        High       0.79      0.73      0.76        30

    accuracy                           0.83       125
   macro avg       0.81      0.80      0.80       125
weighted avg       0.83      0.83      0.83       125



In [50]:
from sklearn.metrics import f1_score

print("Macro F1:", f1_score(y_test, pred, average="macro"))
print("Weighted F1:", f1_score(y_test, pred, average="weighted"))


Macro F1: 0.8044232153373029
Weighted F1: 0.830714126807564


In [44]:
new_claim = "The wind farm has a generating capacity of 759 megawatts and will supply almost 3% of electricity demand in the Netherlands"
print(clf.predict([new_claim]))

ValueError: Expected 2D array, got 1D array instead:
array=['The wind farm has a generating capacity of 759 megawatts and will supply almost 3% of electricity demand in the Netherlands'].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.